# H2O AutoML
**Extended — Pattern Recognition for the Rest of Us**

> Automated model training, stacking, and built-in explainability (SHAP + variable importance) using H2O.ai — runs free in Colab.

**Connection to Model Explainability:** H2O AutoML trains dozens of models automatically *and* produces SHAP values, partial dependence plots, and variable importance out of the box — making it one of the most practical ways to combine AutoML with explainability in a single workflow.

**How to use this notebook:**
- Run cells top to bottom (Runtime → Run all, or Shift+Enter per cell)
- Fill in the `# YOUR CODE HERE` sections in the exercise
- Complete the quiz at the bottom
- Save a copy to your GitHub fork: File → Save a copy in GitHub

In [ ]:
# Titanic via seaborn (bundled, no network needed in Colab)
import seaborn as sns
titanic = sns.load_dataset('titanic')
# Clean up
titanic['sex_num'] = (titanic['sex'] == 'female').astype(int)
titanic = titanic[['survived','pclass','sex_num','age','sibsp','parch','fare']].dropna()
print(f"Titanic: {titanic.shape}  |  Survival rate: {titanic['survived'].mean():.1%}")
titanic.head()

## 🎯 Concept & Intuition

H2O AutoML automates the most time-consuming parts of the modeling workflow:

- **Model training:** fits GBMs, XGBoost, Random Forests, Deep Learning, GLMs, and Stacked Ensembles automatically
- **Hyperparameter tuning:** searches parameter spaces per algorithm without manual grid search
- **Stacking:** builds a meta-learner on top of the best models, often beating any single model
- **Explainability:** generates SHAP values, variable importance, and PDP plots for any model in the leaderboard

Think of it as running LazyPredict + SHAP + cross-validation in one function call — with production-grade models.

**When to use H2O AutoML:**
- You need a strong baseline quickly on a new dataset
- You want to compare model families without writing boilerplate
- You need explainability alongside the model (stakeholder reporting, regulated industries)
- You're benchmarking before committing to a specific model architecture

## 🐍 Setup — Install & Initialize H2O

In [ ]:
# Install H2O (takes ~2 minutes in Colab — run once)
!pip install h2o -q

import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Initialize H2O cluster
# In Colab this starts a local single-node cluster
h2o.init(max_mem_size='4G')
print("H2O version:", h2o.__version__)

## 📊 Load Data

We'll use the classic Titanic dataset — binary classification (survived or not). Swap in any CSV from the `../data/` folder.

## 🤖 Run H2O AutoML

In [ ]:
# Convert to H2O Frame
hf = h2o.H2OFrame(df)

# Mark target as categorical (classification)
hf['Survived'] = hf['Survived'].asfactor()

# Train/test split
train, test = hf.split_frame(ratios=[0.8], seed=42)

# Define predictors and target
target = 'Survived'
predictors = [c for c in hf.columns if c != target]

print(f"Training on {train.nrows} rows, testing on {test.nrows} rows")
print(f"Predictors: {predictors}")

In [ ]:
# Run AutoML
# max_models=15 trains up to 15 models (increase for better results, takes longer)
# seed ensures reproducibility
aml = H2OAutoML(
    max_models=15,
    seed=42,
    max_runtime_secs=120,   # 2-minute cap — increase for larger datasets
    sort_metric='AUC',
    include_algos=['GBM','XGBoost','RandomForest','GLM','StackedEnsemble']
)

aml.train(x=predictors, y=target, training_frame=train)

# View leaderboard — ranked by AUC
lb = aml.leaderboard
print("\n=== AutoML Leaderboard (top 10) ===")
lb.head(rows=10)

## 📈 Evaluate Best Model

In [ ]:
# Get best model
best = aml.leader
print("Best model:", best.model_id)
print("\n=== Performance on Test Set ===")
perf = best.model_performance(test)
print(f"AUC:      {perf.auc():.4f}")
print(f"Accuracy: {perf.accuracy()[0][1]:.4f}")
print(f"Logloss:  {perf.logloss():.4f}")

# Confusion matrix
print("\n=== Confusion Matrix ===")
print(perf.confusion_matrix())

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, auc as sklearn_auc

# Get predictions on test set
preds = best.predict(test).as_data_frame()
test_df = test.as_data_frame()

fpr, tpr, _ = roc_curve(test_df['Survived'].astype(int), preds['p1'])
roc_auc = sklearn_auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#1e5fa8', lw=2, label=f'Best model (AUC = {roc_auc:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=.5, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — H2O AutoML Best Model')
ax.legend(loc='lower right')
ax.grid(True, alpha=.3)
plt.tight_layout()
plt.show()

## 🔬 Explainability — SHAP, Variable Importance & PDP

This is the key tie-in with the **Model Explainability** technique. H2O generates a full explanation report with one call.

In [ ]:
# Full explainability report — generates SHAP, variable importance, PDP
# This is the H2O equivalent of running SHAP + LIME + PDP separately
print("Generating explainability report...")
expl = best.explain(test)
# H2O renders the report inline in Colab automatically

In [ ]:
# Variable importance (standalone plot)
if hasattr(best, 'varimp'):
    varimp = best.varimp(use_pandas=True)
    if varimp is not None:
        fig, ax = plt.subplots(figsize=(7, 4))
        varimp_top = varimp.head(8)
        bars = ax.barh(varimp_top['variable'], varimp_top['scaled_importance'],
                       color='#1e5fa8', edgecolor='white', linewidth=.5)
        ax.set_xlabel('Scaled Importance')
        ax.set_title('Variable Importance — H2O Best Model')
        ax.invert_yaxis()
        for bar, val in zip(bars, varimp_top['scaled_importance']):
            ax.text(val + .005, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=10, color='#1a1d23')
        ax.grid(True, axis='x', alpha=.3)
        plt.tight_layout()
        plt.show()
else:
    print("Variable importance not available for this model type (e.g. Stacked Ensemble).")
    print("Try: specific_model = h2o.get_model(aml.leaderboard[1,'model_id'])")

In [ ]:
# SHAP summary — individual prediction contributions
# Works on GBM, XGBoost, Random Forest models
try:
    # Get a non-ensemble model for SHAP (ensembles don't support SHAP directly)
    gbm_id = None
    for i in range(aml.leaderboard.nrows):
        mid = aml.leaderboard[i, 'model_id']
        if 'GBM' in mid or 'XGBoost' in mid:
            gbm_id = mid
            break

    if gbm_id:
        gbm_model = h2o.get_model(gbm_id)
        shap_values = gbm_model.shap_summary_plot(test)
        print(f"SHAP summary for: {gbm_id}")
    else:
        print("No GBM/XGBoost in leaderboard — adjust include_algos and rerun.")
except Exception as e:
    print(f"SHAP not available for this model: {e}")
    print("Tip: Run with include_algos=['GBM','XGBoost'] to ensure SHAP-compatible models.")

## 🔗 Connection: H2O AutoML ↔ Model Explainability

H2O AutoML and the standalone Model Explainability technique (SHAP/LIME) are complementary:

| | H2O AutoML | Standalone SHAP/LIME |
|---|---|---|
| **Best for** | Benchmarking many models quickly | Deep-dive on one specific model |
| **SHAP support** | Built-in for tree models | Full control, any model |
| **Stakeholder reports** | Auto-generated HTML | Custom plots |
| **Control** | Less (automated) | Full |

**Recommendation:** Use H2O AutoML to find the best model family, then apply standalone SHAP for the final stakeholder deliverable.

## 💼 Exercise

1. **Swap the dataset** — load a CSV from `../data/` or any sklearn dataset
2. **Change the target** — try a regression problem (set `asfactor()` for classification, leave as-is for regression)
3. **Increase `max_models`** to 25 — does the leaderboard change significantly?
4. **Compare model types** — find the best GBM vs the best Stacked Ensemble and compare AUC
5. **Generate a stakeholder memo** — using the variable importance and SHAP plots, write 3 sentences explaining which features drive survival predictions and what a decision-maker should know

```python
# YOUR CODE HERE
```

In [ ]:
# Exercise workspace
# YOUR CODE HERE

### Reflection

*In 2–3 sentences: when would you choose H2O AutoML over manually training sklearn models? What does the explainability output tell you that a single accuracy score doesn't?*

_..._

## 📚 Resources

- [H2O AutoML docs](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/automl.html)
- [H2O.ai free tutorials](https://www.h2o.ai/resources/)
- [H2O explain() documentation](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/explain.html)
- [ISLP Model Explainability notebook](../model-explainability.ipynb) — companion technique

## 🧪 Quiz

Answer in the code cell below — run it to check.

In [ ]:
# @title 📝 Quiz — H2O AutoML
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does H2O AutoML train automatically (name 3 model types)?
# @markdown **Q2:** What is a Stacked Ensemble and why does it often outperform single models?
# @markdown **Q3:** Name two explainability outputs H2O's explain() produces
# @markdown **Q4:** When would you use H2O AutoML vs standalone SHAP/LIME?
# @markdown **Q5:** One real-world use case where AutoML + explainability matters most

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_="H2O AutoML"
# @title 📋 Get AI Feedback on Your Quiz
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2:** Click ▶ Run — your answers are formatted as a prompt
# @markdown
# @markdown **Step 3:** Copy the output → click the **✨ Gemini button** in the Colab toolbar (top right) → paste → send
# @markdown
# @markdown **Step 4:** Keep the conversation going — ask Gemini to explain any output,
# @markdown code block, or chart from this notebook for deeper understanding

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    nd = sum(1 for v in answers.values() if str(v).strip())

    sep = "\u2550" * 62

    print(sep)
    print(f"  Student  : @{GITHUB_USERNAME.strip()}" if GITHUB_USERNAME.strip()
          else "  Student  : (set GITHUB_USERNAME above)")
    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answered : {nd}/{len(answers)} questions")
    print(sep)
    print()
    print("  \u2728  COPY THE PROMPT BELOW INTO GEMINI  \u2728")
    print("  (click the Gemini button in the Colab toolbar, top right)")
    print()
    print("\u2500" * 62)
    print()
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  - Say CORRECT, PARTIAL, or INCORRECT")
    print("  - One sentence: what I got right, or exactly what concept I should review")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - One specific thing to study if I struggled")
    print()
    print("After grading, I may paste specific code outputs or charts from this")
    print("notebook and ask follow-up questions — please help me understand them.")
    print()
    print("\u2500" * 62)
    print()
    print("  After pasting, keep the Gemini chat open and ask things like:")
    print("  \"Why did my SHAP waterfall plot show X?\"")
    print("  \"Can you explain what this output means: [paste output]\"")
    print("  \"I got a different result than the notebook — what went wrong?\"")
    print()
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork")
    print(sep)


---
*Pattern Recognition for the Rest of Us · H2O AutoML · Extended*

*Tip: Shut down the H2O cluster when done to free memory:*
```python
h2o.shutdown(prompt=False)
```